In [34]:
from importlib.metadata import distribution
from locale import normalize

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
import re

In [ ]:
LING_SPAM_PATH = './datasets/Ling-Spam.csv'
NAZARIO_PATH = './datasets/Nazario.csv'
NIGERIAN_FRAUD_PATH = './datasets/Nigerian_Fraud.csv'
SPAMASSASSIN_PATH = './datasets/SpamAssasin.csv'

REQUIRED_COLUMNS = ['body', 'label']

DATASETS = {
    "Ling-Spam": LING_SPAM_PATH,
    "Nazario": NAZARIO_PATH,
    "Nigerian_Fraud": NIGERIAN_FRAUD_PATH,
    "SpamAssasin": SPAMASSASSIN_PATH
}

# Set random seed for reproducibility
np.random.seed(42)

# Load the dataset individually
df = pd.read_csv(DATASETS["Ling-Spam"])

#df.head()

In [ ]:
# Check for missing values in each dataset and the different datatypes of the columns
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df.isnull().sum()} + \n")
    print(f"{df.info()}" + "\n")

    # Body and label columns are the only common columns across all datasets, so we will only keep those columns for further analysis

In [ ]:
# Check the distribution of labels in each dataset
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df['label'].value_counts(dropna=False)}" + "\n")

In [ ]:
# Cleaning the SpamAssassin dataset
def clean_spam_assassin(spam_assassin_df):
    if 'body' in spam_assassin_df.columns:
        spam_assassin_df['body'] = spam_assassin_df['body'].fillna('')
    return spam_assassin_df


# Cleaning the Ling-Spam dataset
def clean_ling_spam(ling_spam_df):
    if 'message' in ling_spam_df.columns:
        ling_spam_df = ling_spam_df.rename(columns={'message': 'body'})
    return ling_spam_df



In [31]:
raw_datasets = {}
cleaned_datasets = {}
def preprocess_datasets():

    for name, path in DATASETS.items():
        df = pd.read_csv(path)
        raw_datasets[name] = df

    for name, df in raw_datasets.items():
        df = clean_spam_assassin(df) if name == "SpamAssasin" else df
        df = clean_ling_spam(df) if name == "Ling-Spam" else df

        for column in REQUIRED_COLUMNS:
            if column not in df.columns:
                raise ValueError(f"Column '{column}' is missing in dataset '{name}'")

            cleaned_datasets[name] = df[REQUIRED_COLUMNS].copy()

    for name, df in cleaned_datasets.items():
        print(f"Dataset: {name}")
        print(f"{df.info()}" + "\n")

preprocess_datasets()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    2893 non-null   object
 1   label   2893 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 45.3+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    1565 non-null   object
 1   label   1565 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 24.6+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3332 entries, 0 to 3331
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    3332 non-null   object
 1   label   3332 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 52.2+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5809 entries, 0 to 5808
Data 

In [42]:
# Training datasets for cross-dataset evaluation design using leave-one-dataset-out evaluation startegy. The training datasets are created by combining three datasets and the testing dataset is the remaining one.

# 1. Train on Nigerian_Fraud + SpamAssasin + Ling-Spam, test on Nazario

# 2. Train on Nigerian_Fraud + SpamAssasin + Nazario, test on Ling-Spam

# 3. Train on Nigerian_Fraud + Ling-Spam + Nazario, test on SpamAssasin

# 4. Train on SpamAssasin + Ling-Spam + Nazario, test on Nigerian_Fraud

combined_datasets = pd.concat([df.assign(source=name) for name, df in cleaned_datasets.items()])

logo = LeaveOneGroupOut()
results = []
train_df = None
val_df = None

for train_idx, test_idx in logo.split(combined_datasets, combined_datasets['label'], groups=combined_datasets['source']):
    training_pool = combined_datasets.iloc[train_idx]
    test_df = combined_datasets.iloc[test_idx]
    held_out_dataset_name = test_df['source'].iloc[0]

    # validation split for training pool
    train_df, val_df = train_test_split(training_pool, test_size=0.2, random_state=42, stratify=training_pool['label'].astype(str) + '_' + training_pool['source'])

    print(f"Training on: {training_pool['source'].unique()} | Testing on: {held_out_dataset_name}")

train_df



#combined_datasets

Training on: ['Nazario' 'Nigerian_Fraud' 'SpamAssasin'] | Testing on: Ling-Spam
Training on: ['Ling-Spam' 'Nigerian_Fraud' 'SpamAssasin'] | Testing on: Nazario
Training on: ['Ling-Spam' 'Nazario' 'SpamAssasin'] | Testing on: Nigerian_Fraud
Training on: ['Ling-Spam' 'Nazario' 'Nigerian_Fraud'] | Testing on: SpamAssasin


,body,label,source
884,Click here to read more \t \t Dear...,1,Nazario
1143,this text is also available in german ( see be...,0,Ling-Spam
2133,I have a new email address!You can now email m...,1,Nigerian_Fraud
24,"hello , there is someone who knows where can i...",0,Ling-Spam
512,"the publisher in rutland , vt and tokyo is cal...",0,Ling-Spam
...,...,...,...
2876,the titus project & the indogermanische gesell...,0,Ling-Spam
1113,DEAR SIR/MADAM This letter may come to you as ...,1,Nigerian_Fraud
1531,international symposium on computer learner co...,0,Ling-Spam
716,Request for Quotation ; Trade...,1,Nazario
